In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import warnings

# 1. Setup Data
celsius    = np.array([-40, -10,  0,  8, 15, 22, 38],  dtype=float)
fahrenheit = np.array([-40,  14, 32, 46, 59, 72, 100], dtype=float)

X = torch.tensor(celsius, dtype=torch.float32).view(-1, 1)
y = torch.tensor(fahrenheit, dtype=torch.float32).view(-1, 1)

# 2. Define and Train Model
class TempConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(1, 1)
        
    def forward(self, x): # <--- The variable name here is 'x'
        return self.layer(x)

model = TempConverter()
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

print("Training started...")
for epoch in range(800):
    prediction = model(X)
    loss = loss_fn(prediction, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 3. PREPARE FOR EXPORT
model.eval() 

# FIX: The key here MUST be 'x' to match the 'forward(self, x)' above
batch_dim = torch.export.Dim("batch", min=1)
dynamic_shapes = {"x": {0: batch_dim}} 

# 4. EXPORT
torch.onnx.export(
    model, 
    (torch.randn(1, 1),), 
    "trained_temp_model.onnx",
    input_names=['celsius'],    # This is what the LABEL will be in the file
    output_names=['fahrenheit'], # This is what the LABEL will be in the file
    dynamic_shapes=dynamic_shapes,
    opset_version=18,
    dynamo=True
)

print("\nSuccess! Model trained and exported to: trained_temp_model.onnx")

# 5. TEST
import onnxruntime as ort
session = ort.InferenceSession("trained_temp_model.onnx")
test_val = np.array([[100.0]], dtype=np.float32)

# Use 'celsius' here because that's what we named the input_names in step 4
prediction = session.run(['fahrenheit'], {'celsius': test_val})

print(f"Result for 100°C: {prediction[0][0][0]:.2f}°F")

Training started...
[torch.onnx] Obtain model graph for `TempConverter([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TempConverter([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

Success! Model trained and exported to: trained_temp_model.onnx
Result for 100°C: 211.72°F
